# Sorting lists with MergeSort
Last week we looked at a simple algorithm for sorting a list of integer numbers. One drawback of the InsertionSort algorithm is that (on average) the number of comparisons required to sort a list of $n$ integers grows in proportion to the length of the list $n$. Hence, the computational complexity is $\mathcal{O}(n^2)$, and sorting large lists becomes prohibitively expensive. In this notebook we will look at a different sorting algorithm, which uses a Divide-and-Conquer strategy to reduce the computational complexity to $\mathcal{O}(n\log_2(n))$. In an abstract sense we used a similar strategy in the FastPower algorithm: there we reduced the original problem to a problem of approximately half the size. As in that example, we will implement the MergeSort algorithm recursively in Python.

<hr style="height: 2px">

### What you will learn
In this notebook we will cover the following topics:

* Sorting with splitting and merging
* Recursive implementation of MergeSort
* Analysis of the computational complexity

<hr style="height: 2px">

*&copy; Eike Mueller, University of Bath 2019-2023. These notes are copyright of Eike Mueller, University of Bath. They are provided exclusively for educational purposes at the University and are to be downloaded or copied for your private study only. Further distribution, e.g. by upload to external repositories, is prohibited.*

## Sorting by splitting and merging
To illustrate the idea behind MergeSort, let us look at an example first. Let's imagine that we want to sort the list $[3,7,5,2,6,4,1]$ with $n=7$ entries. As the following diagram shows, this can be done in two stages:
1. Recursively split the list into smaller lists until each list only contains one element $$[3,7,5,2,6,4,1]\quad\mapsto\quad[3,7,5], [2,6,4,1]\quad\mapsto\quad[3], [7,5], [2,6], [4,1]\quad\mapsto\quad[3],[7],[5],[2],[6],[4],[1].$$
2. Merge the lists back together such that each of the merged lists are sorted. For this, merge two lists at a time $$[3],[7],[5],[2],[6],[4],[1]\quad\mapsto\quad[3], [5,7], [2,6], [1,4]\quad\mapsto\quad[3,5,7], [1,2,4,6]\quad\mapsto\quad[1,2,3,4,5,6,7].$$

For example, in the penultimate step we merge the two sorted lists $[2,6]$ and $[1,4]$ to obtain a new sorted list $[1,2,4,6]$. It might be easier to visualise the process in the following diagram which shows the symmetry between the split- and the merge-stage:

![Sorting the list $[3,7,5,2,6,4,1]$ with `MergeSort`](figures/MergeSort.svg)

## Merging
To implement this algorithm, we need to be able to merge two lists $a^{(1)}$ and $a^{(2)}$, which we assume to be already sorted (note in particular that a list of length 1 is trivially sorted). We will see below why this is a sensible assumption. Given these two lists $a^{(1)}$ and $a^{(2)}$ we can obtain a sorted list $a=\textsf{Merge}(a^{(1)},a^{(2)})$ by repeatedly picking elements from either $a^{(1)}$ or $a^{(2)}$, whichever is smaller. This is illustrated in the following diagram which shows the process for the final step in the example above:

![Merging the two lists $[3,5,7,8]$ and $[1,2,4,6,9]$ with `Merge`](figures/MergeSortTwoLevel.png)

Merge generally, an algorithm for merging $a^{(1)}$ and $a^{(2)}$ to obtain a sorted list $a=\textsf{Merge}(a^{(1)},a^{(2)})$ can be written down explicitly in pseudocode as follows:

### Algorithm: Merge two sorted lists
**Input**: sorted lists $a^{(1)}$ and $a^{(1)}$
**Output**: sorted list $a=\textsf{Merge}(a^{(1)},a^{(2)})$, containing all elements from $a^{(1)}$ and $a^{(2)}$
1. Set $a=[]$
1. **while** neither $a^{(1)}$ nor $a^{(2)}$ are empty **do**
1. $~~~~$**if** $a_{0}^{(1)}< a_{0}^{(2)}$ **then**
1. $~~~~~~~~$Remove first element of $a^{(1)}$ and append it to $a$
1. $~~~~$**else**
1. $~~~~~~~~$Remove first element of $a^{(2)}$ and append it to $a$
1. $~~~$**end if**
1. **end while**
1. Append the remaining elements of $a^{(1)}$ to $a$
1. Append the remaining elements of $a^{(2)}$ to $a$

Note that after executing the while-loop, at least one of the lists $a^{(1)}$ or $a^{(2)}$ will be empty, so only one of the final two lines will actually do something.

To implement the algorithm in Python, we need the following list operations:
1. To **append** an element $x$ to a list $a=[a_0,a_1,...,a_{n-1}]$ we can use `a.append(x)`. After this, the list has the form $[a_0,a_1,...,a_{n-1},x]$
2. To **remove** the the $k$-th element from the list $a=[a_0,a_1,...,a_{n-1}]$, we can use `a.pop(k)`. This will return the value of the $k$-th element, which will be removed from the list $a$ after calling the `pop` method. For example:

In [ ]:
a_list = [1,2,3,4,5]
element_two = a_list.pop(2) # extract the element with index 2 and remove it from the list
print(element_two) 
print(a_list) # The list is now missing the element that has been pop'ed.

3. To **check whether a list is not empty**, we simply check whether its length is larger than one, i.e. `len(a)>0`.
4. To **join** two lists $a=[a_0,a_1,...,a_{n-1}]$ and $b=[b_0,b_1,...,b_{n-1}]$, we use the `+` operator. `a+b` will return a new list $c=[a_0,a_1,...,a_{n-1},b_0,b_1,...,b_{n-1}$

The following code illustrates these operations (try it out and make sure you understand the output):

In [ ]:
# Create two lists
a = [1,2,3]; b=[4,5,6,7]
print('a = ',a,', b = ',b)

# Join the two lists a and b
print('a+b = ',a+b)

# Append an element to the list a
print('a (before appending) = ',a)
a.append(111)
print('a (after appending)  = ',a)

# Remove the element with index 1 from a
print('a (before removing element with index 1) = ',a)
a1 = a.pop(1)
print('element with index 1 = ',a1)
print('a (after removing first)  = ',a)

Based on these list operations we can now implement a Python function for merging two sorted lists:

In [ ]:
def merge(a1,a2):  
    '''Merge two sorted lists a1 and a2 and return merged list
    
    input:
        a1 = first sorted list
        a2 = second sorted list
    output:
        sorted list containing all elements from a1 and a2
    '''
    a = []
    while (len(a1)>0) and (len(a2)>0):
        # Pick the smaller of the two elements
        if a1[0] < a2[0]:
            a.append(a1.pop(0))
        else:
            a.append(a2.pop(0))
    # Append the remaining elements of either a1 or a2 to the final list
    # (only one of these lines will actually do something)
    a += a1
    a += a2
    return a

Let's try this out by creating two (sorted) lists $a^{(1)} = [1,4,6,8]$ and $a^{(2)}=[2,3,7,9]$ and merging them with the `merge()` function:

In [ ]:
a1 = [1,4,6,8]; a2 = [2,3,7,9]
merge(a1,a2)

What happens to the lists after the call? Do we need to worry about this?

## Recursive implementation of MergeSort
Now that we know how to implement to merge function, writing down the MergeSort algorithm recursively is actually very easy:

### Algorithm: merge_sort (recursive implementation)
**Input**: unsorted list $a$
**Output**: sorted list containing all elements of $a$
1. $n=\text{length(a)}$
1. **if** $n\le1$ **then**
1. $~~~~$**return** $a$
1. **else**
1. $~~~~$ Set $n^*=\lfloor n/2\rfloor$
1. $~~~~$ Set $a^{(1)}\mapsto [a_0,a_1,...,a_{n^*-1}]$
1. $~~~~$ Set $a^{(2)}\mapsto [a_{n^*},...,a_{n-1}]$
1. $~~~~$ **return** $\text{merge}(\text{merge_sort}(a^{(1)}),\text{merge_sort}(a^{(2)}))$
1. **end if**

Lines 6 and 7 implement the splitting stage of the algorithm, whereas the call to merging of the two lists happens in line 8, which also contains the recursive call. Since (for large $n$) $n^*=\lfloor n/2\rfloor\approx n/2$,  the lists $a^{(1)}$ and $a^{(2)}$ are approximately half as long as the original list $a$, and it is for this reason that the method can be considered a divide-and-conquer algorithm.

Why does the algorithm work? First of all by construction the merge function will always return a sorted list, which implies that the above merge_sort function must also return a sorted list (if the list that is passed to merge_sort is of length $n=1$ then trivially also a sorted list is returned). Further, since merge gets passed the output of merge_sort in line 8, the assumption that the two lists passed to that function are sorted is also satisfied.
The only thing that could still go wrong is that the merge_sort function never terminates. This, however, can not happen since in each recursive call the size of the list is reduced by splitting it in half.

The first part of the argument might appear circular, but it is not. To see this, the proof can be made more rigorous by an inductive argument. For this we first prove that the algorithm works for lists of length $1$, then use this to show that it works for lists of length $2$, then for lists of length $3$ and so on. We will not pursue this further here.

To implement the algorithm in Python, recall that if $n$ is an integer, then `n//2` returns $\lfloor n/2\rfloor$. Further, using slicing, the lists $a^{(1)}$ and $a^{(2)}$ are given by `a[:n//2]` and `a[n//2:]`.

In [ ]:
def merge_sort(a):
    '''Sort the list a in ascending order and return sorted list
    
    input:
        a = unsorted list
    output:
        sorted list containing all elements from a
    '''
    n = len(a)
    if (n <= 1):
        return a
    else:
        a1 = a[:n//2]
        a2 = a[n//2:]
        return merge(merge_sort(a1),merge_sort(a2))

Let's try this out for a short list:

In [ ]:
a = [3,1,6,4,7,11,8,2]
merge_sort(a)

## Computational complexity

A natural question is how long it takes to execute the MergeSort algorithm for a list of length $n$. As usual, we are interested in how this cost grows for $n\gg 1$. First, observe that the cost for merging two lists $a^{(1)}$ (length $n^{(1)}$) and $a^{(2)}$ (length $n^{(2)}$) is proportional to $n=n^{(1)}+n^{(2)}$, since we need to inspect $n$  elements and considering each element takes some fixed time. Denoting this cost by $C_{\text{merge}}(n)$ we have
$$
C_{\text{merge}}(n) = \tau\cdot n
$$
for some constant $\tau$. When we execute the Mergesort algorithm for a list of length $n$, two things can happen: if $n==1$, we exit, and calling the function will incur some fixed cost $\tau_0$. If $n>1$, we need to call merge_sort on two lists of lenth $\lfloor n/2\rfloor$ and $n-\lfloor n/2\rfloor$ and then merge two two lists. This implies that the cost $C_{\text{merge_sort}}(n)$ for calling merge_sort on a list of length $n>1$ is
$$
\begin{aligned}
C_{\text{merge_sort}}(n) &= C_{\text{merge}}(n) + C_{\text{merge_sort}}(\lfloor n/2\rfloor) + C_{\text{merge_sort}}(n-\lfloor n/2\rfloor)\\
&= \tau\cdot n + C_{\text{merge_sort}}(\lfloor n/2\rfloor) + C_{\text{merge_sort}}(n-\lfloor n/2\rfloor)
\end{aligned}
$$
and
$$
C_{\text{merge_sort}}(1) = \tau_0
$$
It is now natural to assume that the cost increases monotonically in $n$. If we set $k=\lceil \log_2(n)\rceil\in\mathbb{N}$, i.e. assume that $2^{k-1}<n\le 2^k$, then we find that
$$
C_{\text{merge_sort}}(n) \le \underbrace{C_{\text{merge_sort}}(2^k) = \tau\cdot 2^k + 2\cdot C_{\text{merge_sort}}(2^{k-1})}_{(*)}
$$
The recursion in $(*)$ is solved by
$$
C_{\text{merge_sort}}(2^k) = (\tau_0 + \tau\cdot k)2^k=\mathcal{O}(2^k\cdot k)
$$
This finally implies that the computational complexity of MergeSort is
$$
C_{\text{merge_sort}}(n) = \mathcal{O}(n\log_2 n)
$$

To put this into perspective note that:
* The cost $C_{\text{insertion_sort}}(n)$ of InsertionSort is
  - $\mathcal{O}(n^2)$ in the worst case (a completely unsorted list of descending numbers), but
  - $\mathcal{O}(n)$ in the best case (an already sorted list of increasing numbers), and
  - $\mathcal{O}(n^2)$ if the list $a$ is in random order.
* Unless we make additional assumption, $\mathcal{O}(n\log_2n)$ is usually the best computational complexity that can be achieved with a sorting algorithm
* The logarithmic term is typical for divide-and-conquer algorithms, we will see why in next week's lecture

Using the same arguments as above it is also possible to show that $C_{\text{merge_sort}}=\Omega(n\log_2(n))$ and thus $$C_{\text{merge_sort}}=\Theta(n\log_2(n)).$$

## Measured runtimes
Let's look at some measured runtimes for lists of increasing length. As for InsertionSort, this can be done with the following code which uses the numpy function `random.shuffle` to randomly permute an entire list:

In [ ]:
import numpy as np
np.random.seed(2141257)

for n in (1000,2000,4000):
    a_list = list(range(n))
    np.random.shuffle(a_list)
    %timeit merge_sort(a_list)

If we compare this code to the corresponding code in the [notebook on InsertionSort](../week09/lecture_09b_Sorting.ipynb) we find that MergeSort is significantly faster. Doubling the problem size $n$ results in a much weaker growth in the runtime, which increases by a factor slightly larger than two when going from $n$ to $2n$.

### Comparison of MergeSort and InsertionSort
Let us now look at this comparison more systematically. To produce the following graph, both MergeSort (blue) and InsertionSort (red) were used to sort lists that contained random permutations of the sequence $0,1,2,...,n-1$ for different values of $n$. In each case, times were averaged over several different permutations to obtain an average time $t=t(n)$ for each value of $n$. The corresponding times for the highly-optimised built-in Python function [sorted()](https://docs.python.org/3/howto/sorting.html) are also shown in red.

![Average runtime of MergeSort for problems of different size $n$](figures/sorting_runtime_total.png)

On my computer, sorting a list with 30,000 integers takes less than $6$ seconds with MergeSort. While InsertionSort works reasonably well for small lists, its runtime increases rapidly as $n$ becomes larger.

To make the growth in runtime more explicit, we also plot the time per element, i.e. $t/n$ as in the [notebook on InsertionSort](../week09/lecture_09b_Sorting.ipynb). Since we expect a $\mathcal{O}(\log_2(n))$ increase in this quantity, the horizontal axis is chosen to be logarithmic.

![Average runtime per element of MergeSort() and sorted() for problems of different size $n$](figures/sorting_runtime_per_element_log.png)

As can be seen from the plot, the measured data for MergeSort() is consistent with $t=\alpha n+\beta n\log_2(n)$, as expected from the complexity analysis above.

### Technical aside
Strictly speaking, the graphs above were produced with a slightly modified implementation of the `merge()` function, in which the lists `a1` and `a2` are first converted to the Python [deque](https://docs.python.org/3/library/collections.html#collections.deque) datatype. This is because calling `pop()` on a list of length $n$ actually incurs a cost of $\mathcal{O}(n)$. You can think of a deque as a slightly modified version of a list which allows you to remove the first element with `pop()` at a fixed $\mathcal{O}(1)$ cost. For the purpose of this course you can ignore this subtlety, and use the Python code given above.

## Check your understanding
### Exercise 1
Assume that we have two lists $b=[b_0,b_1,b_2,\dots,b_k]$ and $c=[c_0,c_1,c_2,\dots,c_\ell]$. We assume that $b_i\le c_j$ for all $0\le i\le k$ and $0\le j\le \ell$, but we do **not** assume that either of the two lists is sorted.

What does $\text{merge}(b,c)$ return?

1. The list $[b_0,b_1,b_2,\dots,b_k,c_0,c_1,c_2,\dots,c_\ell]$.
2. A sorted list which contains all elements from $b$ and $c$ (in other words, the merge function behaves as it would if $b$ and $c$ were sorted).
3. A list $[\tilde{b}_0,\tilde{b}_1,\tilde{b}_2,\dots,\tilde{b}_k,\tilde{c}_0,\tilde{c}_1,\tilde{c}_2,\dots,\tilde{c}_\ell]$ where the list $[\tilde{b}_0,\tilde{b}_1,\tilde{b}_2,\dots,\tilde{b}_k]$ is obtained by sorting the list $b$ and the list $[\tilde{c}_0,\tilde{c}_1,\tilde{c}_2,\dots,\tilde{c}_\ell]$ is obtained by sorting the list $c$.
4. We can not predict what $\text{merge}(b,c)$ returns in this case.

Answer 1 is correct: The if-statement in line 3 is always true, so the algorithm will extract the elements from list $b$ one by one and append them to the merged list in the same order in which they appear in $b$. Since list $b$ is empty, line 9 will not be executed. Finally, line 10 will append the elements of $c$ to the merged list in the order in which they appear.

### Exercise 2
Assume that the list $a$ of length $n$ is already sorted and recall that the computational complexity of InsertionSort is $\mathcal{O}(n)$ instead of $\mathcal{O}(n^2)$ in this case. Is it possible to also derive a tighter bound on the computational complexity of MergeSort if $a$ is sorted (you might want to consider the case $n=2^k$, $k\in\mathbb{N}$ for simplicity)?

1. Yes, it is $\mathcal{O}(n)$
2. Yes, it is $\mathcal{O}(\log_2(n))$
3. No, the computational complexity is still $\mathcal{O}(n\log_2(n))$

In contrast to InsertionSort, the computational complexity of MergeSort is always $\mathcal{O}(n\log_2(n))$. This is because the merge operation always requires inserting $\mathcal{O}(n)$ elements into the merged list, so it incurs a cose $\mathcal{O}(n)$.